# Gradient Boosting

## Overview

Gradient boosting builds trees sequentially, each correcting the residual errors of the previous. Unlike random forest (parallel, high-variance trees), boosting uses shallow trees (weak learners) combined additively.

**Implementations:**

| Library | Notes |
|---|---|
| `sklearn.GradientBoostingClassifier` | Pure Python, slower, good baseline |
| `XGBoost` | Fast, built-in regularisation, widely used |
| `LightGBM` | Fastest for large data, leaf-wise growth |
| `HistGradientBoostingClassifier` | sklearn native, handles missing values |

**Key hyperparameters:** `n_estimators`, `learning_rate`, `max_depth` (typically 3–6 for boosting), `subsample`, `colsample_bytree`.

Small `learning_rate` + more trees generally beats large `learning_rate` + fewer trees, but requires more computation.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance
from scipy.special import expit

rng = np.random.default_rng(42)
n = 500
elevation  = rng.uniform(50, 400, n)
nitrate    = rng.gamma(2, 2, n)
phosphorus = rng.gamma(1.5, 1.5, n)
ph         = rng.normal(7.2, 0.5, n)
log_odds   = -2 + 0.004*elevation - 0.25*nitrate + 0.4*ph - 0.1*phosphorus
label = (expit(log_odds) > 0.5).astype(int)
X = np.column_stack([elevation, nitrate, phosphorus, ph])
feat_names = ["elevation","nitrate","phosphorus","ph"]
X_tr, X_te, y_tr, y_te = train_test_split(X, label, test_size=0.2,
                                            stratify=label, random_state=42)
print(f"Train: {X_tr.shape}, Prevalence: {label.mean():.3f}")

---
## Fitting with HistGradientBoostingClassifier

In [ ]:
# HistGBM: sklearn native, fast, handles missing values natively
hgb = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=4,
    min_samples_leaf=20, random_state=42
)
hgb.fit(X_tr, y_tr)
print(f"Test accuracy: {hgb.score(X_te, y_te):.3f}")
print(f"Test AUC-ROC:  {roc_auc_score(y_te, hgb.predict_proba(X_te)[:,1]):.3f}")
print(classification_report(y_te, hgb.predict(X_te), target_names=["absent","present"]))

---
## Learning Rate vs n_estimators Tradeoff

In [ ]:
configs = [
    (0.3, 100, "#e74c3c"),
    (0.1, 200, "steelblue"),
    (0.05, 500, "#4fffb0"),
]
fig, ax = plt.subplots(figsize=(8,4))
for lr, n_est, color in configs:
    cv = cross_val_score(
        HistGradientBoostingClassifier(learning_rate=lr, max_iter=n_est, random_state=42),
        X_tr, y_tr, cv=5, scoring="roc_auc"
    )
    ax.barh(f"lr={lr}, n={n_est}", cv.mean(),
            xerr=cv.std(), color=color, alpha=0.8, capsize=4)
ax.set_xlabel("CV AUC-ROC"); ax.set_title("Learning Rate vs Trees Tradeoff")
plt.tight_layout(); plt.show()

---
## Early Stopping

In [ ]:
# Use validation fraction to stop adding trees when validation score stops improving
hgb_es = HistGradientBoostingClassifier(
    max_iter=1000, learning_rate=0.05, max_depth=4,
    validation_fraction=0.15, n_iter_no_change=20,
    random_state=42
)
hgb_es.fit(X_tr, y_tr)
print(f"Trees used (early stopping): {hgb_es.n_iter_}")
print(f"Test AUC: {roc_auc_score(y_te, hgb_es.predict_proba(X_te)[:,1]):.3f}")

---
## Feature Importance and ROC

In [ ]:
perm = permutation_importance(hgb, X_te, y_te, n_repeats=30, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=feat_names)
fig, axes = plt.subplots(1,2,figsize=(12,4))
perm_imp.sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Permutation Importance (test set)")
axes[0].set_xlabel("Mean accuracy decrease")
RocCurveDisplay.from_estimator(hgb, X_te, y_te, ax=axes[1], color="#e74c3c")
axes[1].set_title(f"ROC  (AUC={roc_auc_score(y_te, hgb.predict_proba(X_te)[:,1]):.3f})")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using a large learning rate with few trees**  
Large learning rates make each tree step greedily, often overshooting. Small learning rate (0.01–0.1) with more trees and early stopping consistently outperforms large-rate models and is more robust to hyperparameter choices.

**2. Not using early stopping**  
Without early stopping, boosting will eventually overfit as trees are added beyond the optimum. Always set `n_iter_no_change` or monitor a validation curve, and use the iteration count where validation loss is minimised.

**3. Setting max_depth too large for boosting**  
Boosting works best with shallow weak learners (depth 3–6). Deep trees in a boosting context overfit faster and negate the benefit of the sequential correction approach. Random forests tolerate deeper trees much better.

**4. Skipping feature scaling because tree models do not require it**  
While trees do not require scaling, some boosting regularisation schemes (L2 on leaf weights) can be scale-sensitive when features have very different ranges. Scaling is harmless and sometimes beneficial.

**5. Not comparing to a simpler baseline**  
Gradient boosting is complex and easy to overtune. Always compare test performance to logistic regression and random forest before concluding that boosting adds value for your specific dataset.
---
*python_methods_library - Samantha McGarrigle*